In [ ]:
import os
import re
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# CÃ¡c mÃ´ hÃ¬nh nÃ¢ng cao (Advanced Models)
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Äáº£m báº£o thÆ° má»¥c lÆ°u káº¿t quáº£ tá»“n táº¡i theo yÃªu cáº§u cáº¥u trÃºc dá»± Ã¡n
os.makedirs('./results/model_comparison', exist_ok=True)
os.makedirs('./results', exist_ok=True)
print("ThÆ° má»¥c /results/model_comparison Ä‘Ã£ sáºµn sÃ ng.")

# ÄÃƒ Sá»¬A Lá»–I: Cáº­p nháº­t Ä‘Æ°á»ng dáº«n tá»›i thÆ° má»¥c data/processed cá»§a dá»± Ã¡n
train_df = pd.read_csv('../../data/processed/cleaned_train.csv')
test_df = pd.read_csv('../../data/processed/cleaned_test.csv')

# HÃ m loáº¡i bá» cÃ¡c kÃ½ tá»± Ä‘áº·c biá»‡t JSON trong tÃªn cá»™t Ä‘á»ƒ trÃ¡nh lá»—i LightGBM
def sanitize_column_names(df):
    # Thay tháº¿ cÃ¡c kÃ½ tá»± [ ], {, }, :, , vÃ  \" thÃ nh dáº¥u gáº¡ch dÆ°á»›i _
    df.columns = [re.sub(r'[\[\]\,\{\}\:\"]', '_', col) for col in df.columns]
    return df

# Tiáº¿n hÃ nh lÃ m sáº¡ch tÃªn cá»™t cho cáº£ 2 táº­p dá»¯ liá»‡u
train_df = sanitize_column_names(train_df)
test_df = sanitize_column_names(test_df)

# XÃ¡c Ä‘á»‹nh cá»™t Target lÃ  'Price'
target_col = 'Price'

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

# KhÃ´i phá»¥c giÃ¡ trá»‹ thá»±c táº¿ cá»§a Price vá» Ä‘Æ¡n vá»‹ gá»‘c báº±ng expm1 (vÃ¬ trÆ°á»›c Ä‘Ã³ dÃ¹ng log transform)
y_test_actual = np.expm1(y_test)

print("KÃ­ch thÆ°á»›c táº­p Train:", X_train.shape)
print("KÃ­ch thÆ°á»›c táº­p Test:", X_test.shape)
print("ÄÃ£ táº£i dá»¯ liá»‡u thÃ nh cÃ´ng vÃ  xá»­ lÃ½ xong tÃªn cá»™t!")

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Äá»‹nh nghÄ©a 8 mÃ´ hÃ¬nh ngay táº¡i Ä‘Ã¢y Ä‘á»ƒ trÃ¡nh lá»—i NameError
all_models = {
    'Dummy Regressor': DummyRegressor(strategy="mean"),
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(random_state=42),
    'Lasso': Lasso(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
}

# File Ä‘Ã­ch Ä‘á»ƒ lÆ°u cÃ¡c mÃ´ hÃ¬nh Ä‘Ã£ huáº¥n luyá»‡n
model_save_paths = {
    'Random Forest': './results/random_forest_model.pkl',
    'XGBoost': './results/xgboost_model.pkl',
    'LightGBM': './results/lightgbm_model.pkl'
}

# Cáº¥u hÃ¬nh K-Fold 5 splits
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

print("â³ Äang cháº¡y K-Fold Cross Validation (5 splits)... Vui lÃ²ng Ä‘á»£i vÃ i phÃºt!")

# Duyá»‡t qua tá»« Ä‘iá»ƒn mÃ´ hÃ¬nh vá»«a táº¡o riÃªng cho CV
for name, model in all_models.items():
    fold_maes = []

    for train_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)
        pred_log = model.predict(X_val)

        pred_log_clipped = np.clip(pred_log, a_min=0, a_max=25)
        pred_actual = np.expm1(pred_log_clipped)
        y_val_actual = np.expm1(y_val)

        fold_maes.append(mean_absolute_error(y_val_actual, pred_actual))

    mean_cv_mae = np.mean(fold_maes)
    std_cv_mae = np.std(fold_maes)

    cv_results.append({
        'Model': name,
        'Mean CV MAE': mean_cv_mae,
        'Std CV MAE': std_cv_mae
    })
    print(f"âœ… {name} -> Mean CV MAE: {mean_cv_mae:,.2f} | Std: {std_cv_mae:,.2f}")

# Xuáº¥t káº¿t quáº£ ra Excel
df_cv = pd.DataFrame(cv_results).sort_values(by='Mean CV MAE')
df_cv.to_excel('./results/model_comparison/cv_metrics.xlsx', index=False)
print("\n[SUCCESS] ÄÃ£ lÆ°u báº£ng káº¿t quáº£ K-Fold CV vÃ o thÆ° má»¥c results!")

predictions_dict = {}

print("Äang huáº¥n luyá»‡n vÃ  dá»± bÃ¡o cho cÃ¡c mÃ´ hÃ¬nh...")
for name, model in all_models.items():
    print(f"-> Äang cháº¡y: {name}...")
    model.fit(X_train, y_train)

    pred_log = model.predict(X_test)
    pred_log_clipped = np.clip(pred_log, a_min=0, a_max=25)
    pred_actual = np.expm1(pred_log_clipped)

    predictions_dict[name] = pred_actual
    if name in model_save_paths:
        joblib.dump(model, model_save_paths[name])
        print(f"[SAVE] {name} -> {model_save_paths[name]}")

print("\n[SUCCESS] ÄÃ£ hoÃ n thÃ nh dá»± Ä‘oÃ¡n cho cáº£ 8 mÃ´ hÃ¬nh!")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, mean_absolute_error, r2_score

error_analysis_summary = []
cv_results = []

print("======= Káº¾T QUáº¢ K-FOLD CROSS VALIDATION (5 SPLITS) =======\n")

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in all_models.items():
    fold_maes = []

    for train_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)
        pred_log = model.predict(X_val)

        pred_log_clipped = np.clip(pred_log, a_min=0, a_max=25)
        pred_actual = np.expm1(pred_log_clipped)
        y_val_actual = np.expm1(y_val)

        fold_maes.append(mean_absolute_error(y_val_actual, pred_actual))

    mean_cv_mae = np.mean(fold_maes)
    std_cv_mae = np.std(fold_maes)

    cv_results.append({
        'Model': name,
        'Mean CV MAE': mean_cv_mae,
        'Std CV MAE': std_cv_mae
    })

    print(f"ðŸ‘‰ MÃ´ hÃ¬nh: {name}")
    print(f"   - Mean CV MAE:   {mean_cv_mae:,.2f}")
    print(f"   - Std CV MAE:    {std_cv_mae:,.2f}")
    print("-" * 65)

df_cv = pd.DataFrame(cv_results).sort_values(by='Mean CV MAE')
df_cv.to_excel('./results/model_comparison/cv_metrics.xlsx', index=False)
print("\n[SUCCESS] ÄÃ£ lÆ°u báº£ng káº¿t quáº£ K-Fold CV vÃ o thÆ° má»¥c results!\n")

print("======= Káº¾T QUáº¢ RMSE, MAPE, R2 VÃ€ PHÃ‚N TÃCH Lá»–I CHI TIáº¾T (ÄÆ¡n vá»‹: Triá»‡u VNÄ) =======\n")

for name, pred_actual in predictions_dict.items():
    rmse_val = np.sqrt(mean_squared_error(y_test_actual, pred_actual))
    mape_val = mean_absolute_percentage_error(y_test_actual, pred_actual) * 100
    r2_val = r2_score(y_test_actual, pred_actual)
    absolute_errors = np.abs(y_test_actual - pred_actual)

    # Loáº¡i bá» cÃ¡c giÃ¡ trá»‹ NaN/Inf (Ä‘á» phÃ²ng váº«n cÃ²n tÃ n dÆ°)
    valid_errors = absolute_errors[~np.isnan(absolute_errors) & ~np.isinf(absolute_errors)]

    if len(valid_errors) == 0:
        print(f"Cáº¢NH BÃO: MÃ´ hÃ¬nh {name} bá»‹ lá»—i toÃ n NaN/Inf!")
        continue

    mean_err = np.mean(valid_errors)
    median_err = np.median(valid_errors)
    max_err = np.max(valid_errors)
    r2 = r2_score(y_test_actual, pred_actual)

    # Láº¥y Top 10 Max Error
    top_10_max_err = np.sort(valid_errors)[-10:][::-1]

    # Gá»™p 10 con sá»‘ thÃ nh 1 chuá»—i vÄƒn báº£n Ä‘á»ƒ Ä‘Æ°a vÃ o 1 Ã´ Excel
    top_10_str = ", ".join([f"{err:,.0f}" for err in top_10_max_err])

    error_analysis_summary.append({
        'Model': name,
        'RMSE': rmse_val,
        'MAPE (%)': mape_val,
        'R2 Score': r2,
        'Mean Error': mean_err,
        'Median Error': median_err,
        'Max Error': max_err,
        'Top 10 Errors': top_10_str
    })

    print(f"ðŸ‘‰ MÃ´ hÃ¬nh: {name}")
    print(f"   - RMSE:          {rmse_val:,.2f}")
    print(f"   - MAPE:          {mape_val:,.2f}%")
    print(f"   - R2 Score:      {r2_val:.4f}")
    print(f"   - Mean Error:      {mean_err:,.2f}")
    print(f"   - Median Error:    {median_err:,.2f}")
    print(f"   - Max Error:       {max_err:,.2f}")
    print(f"   - Top 10 Max Err:  {top_10_str}")
    print("-" * 65)

# LÆ°u DataFrame ra Excel
df_errors = pd.DataFrame(error_analysis_summary).sort_values(by='R2 Score', ascending=False)
df_errors.to_excel('./results/model_comparison/error_analysis_summary.xlsx', index=False)
print("\n[SUCCESS] ÄÃ£ lÆ°u báº£ng phÃ¢n tÃ­ch bao gá»“m RMSE, MAPE, R2 vÃ  cá»™t Top 10 vÃ o Excel!")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingRegressor, RandomForestRegressor
from sklearn.linear_model import RidgeCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

print("======= Ká»¸ THUáº¬T NÃ‚NG CAO: FEATURE ENGINEERING & STACKING =======\\n")

# 1. Feature Engineering (NhÃ o náº·n Ä‘áº·c trÆ°ng)
X_train['Area_per_Bedroom'] = X_train['Area'] / (X_train['Bedrooms'] + 1)
X_test['Area_per_Bedroom'] = X_test['Area'] / (X_test['Bedrooms'] + 1)

X_train['Is_Large_House'] = (X_train['Area'] >= 150).astype(int)
X_test['Is_Large_House'] = (X_test['Area'] >= 150).astype(int)

print(f"Sá»‘ lÆ°á»£ng Ä‘áº·c trÆ°ng sau khi thÃªm: {X_train.shape[1]}")

# 2. Cáº¥u hÃ¬nh cÃ¡c mÃ´ hÃ¬nh cÆ¡ sá»Ÿ (Base Models) cho Stacking
# LÆ¯U Ã ÄÃƒ FIX Lá»–I: Chá»‘t n_estimators = 300 Ä‘á»ƒ Stacking cháº¡y nhanh vÃ  khÃ´ng Overfit
xgb_base = XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42)
lgbm_base = LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=31, random_state=42, verbose=-1)
rf_base = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)

estimators = [
    ('XGBoost', xgb_base),
    ('LightGBM', lgbm_base),
    ('RandomForest', rf_base)
]

# 3. XÃ¢y dá»±ng Stacking Regressor
print("â³ Äang huáº¥n luyá»‡n mÃ´ hÃ¬nh Stacking (Sáº½ máº¥t khoáº£ng 1-3 phÃºt)...")
stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=RidgeCV(), # DÃ¹ng RidgeCV tá»± Ä‘á»™ng tÃ¬m há»‡ sá»‘ pháº¡t lÃ m mÃ´ hÃ¬nh chá»‘t
    cv=5,
    n_jobs=-1
)

# Huáº¥n luyá»‡n trÃªn táº­p Train
stacking_model.fit(X_train, y_train)
joblib.dump(stacking_model, './results/stacking_model.pkl')
print("[SAVE] stacking_model -> ./results/stacking_model.pkl")
print("âœ… HoÃ n táº¥t huáº¥n luyá»‡n Stacking!")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, mean_absolute_error, r2_score

print("======= ÄÃNH GIÃ MÃ” HÃŒNH VÃš KHÃ Tá»I THÆ¯á»¢NG (STACKING) =======")

# 1. Dá»± Ä‘oÃ¡n trÃªn táº­p Test báº±ng mÃ´ hÃ¬nh Stacking
pred_stacking_log = stacking_model.predict(X_test)

# Clip giÃ¡ trá»‹ log Ä‘á»ƒ trÃ¡nh lá»—i overflow khi expm1
pred_stacking_log_clipped = np.clip(pred_stacking_log, a_min=0, a_max=25)

# KhÃ´i phá»¥c vá» giÃ¡ trá»‹ thá»±c táº¿ (Triá»‡u VNÄ)
pred_stacking_actual = np.expm1(pred_stacking_log_clipped)

# 2. TÃ­nh toÃ¡n cÃ¡c chá»‰ sá»‘ lá»—i cho Stacking
stacking_mae = mean_absolute_error(y_test_actual, pred_stacking_actual)
stacking_rmse = np.sqrt(mean_squared_error(y_test_actual, pred_stacking_actual))
stacking_median_err = np.median(np.abs(y_test_actual - pred_stacking_actual))
stacking_mape = mean_absolute_percentage_error(y_test_actual, pred_stacking_actual) * 100
stacking_r2 = r2_score(y_test_actual, pred_stacking_actual)

print(f"MAE          :      {stacking_mae:,.2f} (Triá»‡u VNÄ)")
print(f"RMSE         :      {stacking_rmse:,.2f} (Triá»‡u VNÄ)")
print(f"Median Error :      {stacking_median_err:,.2f} (Triá»‡u VNÄ)")
print(f"MAPE         :        {stacking_mape:,.2f}%")
print(f"RÂ² Score     :        {stacking_r2:.4f}")
print("-" * 65)

# 3. Cáº­p nháº­t vÃ o báº£ng phÃ¢n tÃ­ch lá»—i Ä‘á»ƒ chuáº©n bá»‹ váº½ biá»ƒu Ä‘á»“ so sÃ¡nh
# Äá»c láº¡i báº£ng káº¿t quáº£ cá»§a 8 mÃ´ hÃ¬nh trÆ°á»›c Ä‘Ã³ tá»« file Ä‘Ã£ lÆ°u
df_errors_existing = pd.read_excel('./results/model_comparison/error_analysis_summary.xlsx')

# Táº¡o dÃ²ng dá»¯ liá»‡u má»›i cho Stacking
stacking_metrics = {
    'Model': 'Stacking Regressor',
    'RMSE': stacking_rmse,
    'MAPE (%)': stacking_mape,
    'R2 Score': stacking_r2,
    'Mean Error': stacking_mae,
    'Median Error': stacking_median_err,
    'Max Error': np.max(np.abs(y_test_actual - pred_stacking_actual)),
    'Top 10 Errors': ", ".join([f"{err:,.0f}" for err in np.sort(np.abs(y_test_actual - pred_stacking_actual))[-10:][::-1]])
}

# ThÃªm Stacking vÃ o báº£ng vÃ  sáº¯p xáº¿p theo R2 Score giáº£m dáº§n
df_all_models = pd.concat([df_errors_existing, pd.DataFrame([stacking_metrics])], ignore_index=True)
df_all_models = df_all_models.sort_values(by='R2 Score', ascending=False)

# LÆ°u láº¡i báº£ng tá»•ng há»£p má»›i bao gá»“m cáº£ Stacking
df_all_models.to_excel('./results/model_comparison/error_analysis_with_stacking.xlsx', index=False)
print("[SUCCESS] ÄÃ£ cáº­p nháº­t káº¿t quáº£ Stacking vÃ o file Excel má»›i!")

# 4. TRá»°C QUAN HÃ“A: Váº½ biá»ƒu Ä‘á»“ so sÃ¡nh RÂ² Score giá»¯a cÃ¡c mÃ´ hÃ¬nh
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

# Táº¡o báº£ng mÃ u, highlight mÃ´ hÃ¬nh Stacking báº±ng mÃ u Äá» ná»•i báº­t, cÃ¡c mÃ´ hÃ¬nh khÃ¡c mÃ u Xanh dá»‹u
colors = ['#e74c3c' if model == 'Stacking Regressor' else '#34495e' for model in df_all_models['Model']]

barplot = sns.barplot(x='R2 Score', y='Model', data=df_all_models, palette=colors)

# Hiá»ƒn thá»‹ giÃ¡ trá»‹ cá»¥ thá»ƒ lÃªn tá»«ng cá»™t
for index, row in df_all_models.iterrows():
    barplot.text(row['R2 Score'] + 0.01, index, f"{row['R2 Score']:.4f}", color='black', va="center", fontweight='bold')

plt.title('So sÃ¡nh RÂ² Score giá»¯a cÃ¡c mÃ´ hÃ¬nh (CÃ ng cao cÃ ng tá»‘t)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('RÂ² Score', fontsize=12)
plt.ylabel('MÃ´ hÃ¬nh', fontsize=12)
plt.xlim(-0.2, 0.6) # Äiá»u chá»‰nh giá»›i háº¡n Ä‘á»ƒ hiá»ƒn thá»‹ rÃµ tá»« Dummy Ä‘áº¿n Stacking
plt.tight_layout()

# LÆ°u biá»ƒu Ä‘á»“ vÃ o thÆ° má»¥c káº¿t quáº£ theo cáº¥u trÃºc dá»± Ã¡n
plt.savefig('./results/model_comparison/model_r2_comparison.png', dpi=300)
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure stacking_model exists; if not, try to train it quickly using available data
if 'stacking_model' not in globals():
    print("Warning: 'stacking_model' not found â€” training a StackingRegressor now...")
    try:
        from sklearn.ensemble import StackingRegressor, RandomForestRegressor
        from sklearn.linear_model import RidgeCV
        xgb_base = XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42)
        lgbm_base = LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=31, random_state=42, verbose=-1)
        rf_base = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)
        estimators = [('XGBoost', xgb_base), ('LightGBM', lgbm_base), ('RandomForest', rf_base)]
        stacking_model = StackingRegressor(estimators=estimators, final_estimator=RidgeCV(), cv=5, n_jobs=-1)
        stacking_model.fit(X_train, y_train)
        print("stacking_model trained successfully.")
    except Exception as e:
        raise RuntimeError(f"stacking_model is not defined and automatic training failed: {e}")

# 1. Dá»± bÃ¡o vÃ  Ä‘Ã¡nh giÃ¡ Stacking
log_pred_stack = stacking_model.predict(X_test)
clipped_log_pred_stack = np.clip(log_pred_stack, a_min=0, a_max=25)
actual_pred_stack = np.expm1(clipped_log_pred_stack)

# ÄÃƒ FIX Lá»–I: DÃ¹ng Ä‘Ãºng biáº¿n y_test_actual
mae_stack = mean_absolute_error(y_test_actual, actual_pred_stack)
rmse_stack = np.sqrt(mean_squared_error(y_test_actual, actual_pred_stack))
mape_stack = mean_absolute_percentage_error(y_test_actual, actual_pred_stack) * 100
r2_stack = r2_score(y_test_actual, actual_pred_stack)
median_err_stack = np.median(np.abs(y_test_actual - actual_pred_stack))

print(f"--- HIá»†U NÄ‚NG MÃ” HÃŒNH VÅ¨ KHÃ Tá»I THÆ¯á»¢NG (STACKING) ---")
print(f"MAE          : {mae_stack:13,.2f} (Triá»‡u VNÄ)")
print(f"RMSE         : {rmse_stack:13,.2f} (Triá»‡u VNÄ)")
print(f"Median Error : {median_err_stack:13,.2f} (Triá»‡u VNÄ)")
print(f"MAPE         : {mape_stack:13.2f}%")
print(f"RÂ² Score     : {r2_stack:13.4f}")
print("-" * 65)

# 2. ÄÃ¡nh giÃ¡ Táº§m quan trá»ng cá»§a Äáº·c trÆ°ng (Permutation Importance)
print("\nâ³ Äang tÃ­nh toÃ¡n Permutation Importance (CÃ³ thá»ƒ máº¥t 1-2 phÃºt)...")
perm_importance = permutation_importance(
    stacking_model, X_test, y_test,
    n_repeats=5, random_state=42,
    scoring='neg_mean_absolute_error'
)

# TrÃ­ch xuáº¥t vÃ  Váº½ Boxplot
sorted_idx = perm_importance.importances_mean.argsort()[-15:] # Láº¥y Top 15

plt.figure(figsize=(12, 8))
sns.boxplot(
    data=perm_importance.importances[sorted_idx].T,
    orient='h',
    palette="viridis"
)
plt.yticks(range(len(sorted_idx)), X_test.columns[sorted_idx])
plt.xlabel("Má»©c Ä‘á»™ sá»¥t giáº£m hiá»‡u nÄƒng (Náº¿u xÃ¡o trá»™n Ä‘áº·c trÆ°ng)", fontsize=12)
plt.title("Top 15 Feature Importance cá»§a Stacking Model", fontsize=14, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('./results/model_comparison/stacking_feature_importance.png')
plt.show()

In [ ]:
# Táº¡o DataFrame gá»™p GiÃ¡ thá»±c táº¿ vÃ  Dá»± Ä‘oÃ¡n cá»§a 8 mÃ´ hÃ¬nh
# Pháº§n 1: MAE theo phÃ¢n khÃºc giÃ¡

df_segments = pd.DataFrame({'Actual_Price': y_test_actual})
for name, pred in predictions_dict.items():
    df_segments[name] = pred

# Äá»‹nh nghÄ©a cÃ¡c má»‘c phÃ¢n khÃºc giÃ¡ (ÄÆ¡n vá»‹: Triá»‡u VNÄ)
bins = [0, 3000, 5000, 10000, 20000, np.inf]
labels = ['1. DÆ°á»›i 3 Tá»·', '2. Tá»« 3 - 5 Tá»·', '3. Tá»« 5 - 10 Tá»·', '4. Tá»« 10 - 20 Tá»·', '5. TrÃªn 20 Tá»·']
df_segments['Segment'] = pd.cut(df_segments['Actual_Price'], bins=bins, labels=labels)

segment_maes = []
for name in predictions_dict.keys():
    for seg in labels:
        mask = df_segments['Segment'] == seg
        if mask.sum() > 0:
            y_true_seg = df_segments.loc[mask, 'Actual_Price']
            y_pred_seg = df_segments.loc[mask, name]
            mae = mean_absolute_error(y_true_seg, y_pred_seg)
            segment_maes.append({'Model': name, 'Segment': seg, 'MAE': mae})

df_segment_mae = pd.DataFrame(segment_maes)
pivot_segment = df_segment_mae.pivot(index='Model', columns='Segment', values='MAE').round(2)
pivot_segment = pivot_segment.sort_values(by='1. DÆ°á»›i 3 Tá»·')
pivot_segment.to_excel('./results/model_comparison/mae_by_segment.xlsx')
print("[SUCCESS] ÄÃ£ lÆ°u báº£ng MAE theo PhÃ¢n KhÃºc GiÃ¡ vÃ o Excel!")
display(pivot_segment)

print("\n--- MAE THEO QUáº¬N/HUYá»†N ---")
district_cols = [col for col in X_test.columns if 'District' in col or 'Quan_' in col or 'Huyen_' in col]

if len(district_cols) > 0:
    districts = X_test[district_cols].idxmax(axis=1)
    districts = districts.str.replace('District_', '').str.replace('Quan_', 'Q').str.replace('Huyen_', 'H')

    df_districts = pd.DataFrame({'Actual_Price': y_test_actual, 'District': districts})
    for name, pred in predictions_dict.items():
        df_districts[name] = pred

    district_maes = []
    for name in predictions_dict.keys():
        for dist in districts.unique():
            mask = df_districts['District'] == dist
            if mask.sum() > 0:
                y_true_dist = df_districts.loc[mask, 'Actual_Price']
                y_pred_dist = df_districts.loc[mask, name]
                mae = mean_absolute_error(y_true_dist, y_pred_dist)
                district_maes.append({'Model': name, 'District': dist, 'MAE': mae})

    df_dist = pd.DataFrame(district_maes)
    pivot_dist = df_dist.pivot(index='Model', columns='District', values='MAE').round(2)
    pivot_dist.to_excel('./results/model_comparison/mae_by_district.xlsx')
    print("[SUCCESS] ÄÃ£ lÆ°u báº£ng MAE theo Quáº­n vÃ o Excel!")
    display(pivot_dist.head())
else:
    print("Cáº¢NH BÃO: KhÃ´ng tÃ¬m tháº¥y cá»™t nÃ o chá»©a tá»« khÃ³a District/Quan/Huyen trong táº­p X_test.")

In [ ]:
# Gá»™p biá»ƒu Ä‘á»“ quan trá»ng nháº¥t vÃ o má»™t cell Ä‘á»ƒ notebook ngáº¯n hÆ¡n
# Pháº§n 1: Top 10 Feature Importance cá»§a 4 mÃ´ hÃ¬nh máº¡nh nháº¥t
advanced_models = {
    'Random Forest': all_models['Random Forest'],
    'XGBoost': all_models['XGBoost'],
    'Gradient Boosting': all_models['Gradient Boosting'],
    'LightGBM': all_models['LightGBM']
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

# Get the original feature names before feature engineering
# These are the features that 'advanced_models' were trained on.
original_feature_names = train_df.drop(columns=[target_col]).columns

for idx, (name, model) in enumerate(advanced_models.items()):
    importances = model.feature_importances_
    # Use the original feature names that match the length of importances
    feature_names_for_plot = original_feature_names

    df_importance = pd.DataFrame({
        'Feature': feature_names_for_plot,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False).head(10)

    sns.barplot(
        x='Importance',
        y='Feature',
        data=df_importance,
        ax=axes[idx],
        palette='viridis',
        hue='Feature',
        legend=False
    )
    axes[idx].set_title(f'Top 10 Feature Importance - {name}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Má»©c Ä‘á»™ quan trá»ng')
    axes[idx].set_ylabel('')

plt.tight_layout()
plt.savefig('./results/model_comparison/top_10_feature_importance.png', dpi=300)
plt.show()

# Pháº§n 2: Actual vs Predicted cho cáº£ 8 mÃ´ hÃ¬nh
fig, axes = plt.subplots(4, 2, figsize=(16, 24))
axes = axes.flatten()

for idx, (name, pred_actual) in enumerate(predictions_dict.items()):
    sample_size = min(1000, len(y_test_actual))
    sample_idx = np.random.choice(len(y_test_actual), sample_size, replace=False)

    actual_sampled = y_test_actual.iloc[sample_idx] if isinstance(y_test_actual, pd.Series) else y_test_actual[sample_idx]
    pred_sampled = pred_actual[sample_idx]

    sns.scatterplot(x=actual_sampled, y=pred_sampled, alpha=0.6, ax=axes[idx], color='teal')

    max_val = max(actual_sampled.max(), pred_sampled.max())
    min_val = min(actual_sampled.min(), pred_sampled.min())
    axes[idx].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='ÄÆ°á»ng lÃ½ tÆ°á»Ÿng (y = x)')

    axes[idx].set_title(f'Actual vs Predicted - {name}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('GiÃ¡ trá»‹ thá»±c táº¿ (Triá»‡u VNÄ)')
    axes[idx].set_ylabel('GiÃ¡ trá»‹ dá»± Ä‘oÃ¡n (Triá»‡u VNÄ)')
    axes[idx].legend()
    axes[idx].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('./results/model_comparison/actual_vs_predicted.png', dpi=300)
plt.show()

# Pháº§n 3: ÄÃ¡nh giÃ¡ clipping trÃªn cÃ¡c mÃ´ hÃ¬nh tiÃªu biá»ƒu
from sklearn.metrics import mean_absolute_error

def evaluate_clipping(model_name, model, X_test_eval, y_test_actual):
    raw_log_pred = model.predict(X_test_eval)
    num_clipped = np.sum(raw_log_pred > 25)

    try:
        with np.errstate(over='ignore'):
            raw_actual_pred = np.expm1(raw_log_pred)
            valid_mask = np.isfinite(raw_actual_pred)
            mae_before = mean_absolute_error(y_test_actual[valid_mask], raw_actual_pred[valid_mask])
            if not np.all(valid_mask):
                print(f"  [!] CÃ³ {np.sum(~valid_mask)} giÃ¡ trá»‹ bá»‹ trÃ n sá»‘ tá»›i vÃ´ cá»±c (inf) trÆ°á»›c clipping.")
    except Exception:
        mae_before = float('inf')

    clipped_log_pred = np.clip(raw_log_pred, a_min=0, a_max=25)
    clipped_actual_pred = np.expm1(clipped_log_pred)
    mae_after = mean_absolute_error(y_test_actual, clipped_actual_pred)

    print(f"--- ÄÃNH GIÃ CLIPPING: {model_name} ---")
    print(f"Sá»‘ lÆ°á»£ng dá»± bÃ¡o bá»‹ Clipping (>25): {num_clipped}")
    print(f"MAE TRÆ¯á»šC Clipping: {mae_before:,.2f} (Triá»‡u VNÄ)")
    print(f"MAE SAU Clipping  : {mae_after:,.2f} (Triá»‡u VNÄ)\n")

# Create a non-feature-engineered X_test specifically for models trained on 59 features
X_test_original = test_df.drop(columns=[target_col])

evaluate_clipping("Linear Regression", all_models["Linear Regression"], X_test_original, y_test_actual)
evaluate_clipping("XGBoost", all_models["XGBoost"], X_test_original, y_test_actual)


In [ ]:
import joblib
# LÆ°u láº¡i mÃ´ hÃ¬nh Random Forest Ä‘Ã£ huáº¥n luyá»‡n Ä‘Ãºng tÃªn biáº¿n
joblib.dump(all_models['Random Forest'], 'results/random_forest_model.pkl')
print("[SAVE] Random Forest -> results/random_forest_model.pkl")

In [ ]:
import os
from pathlib import Path

results_dir = Path('./results')
pkl_files = sorted(results_dir.glob('*.pkl'))

print('======= DANH SÃCH MODEL .PKL ÄÃƒ LÆ¯U =======')
if not pkl_files:
    print('KhÃ´ng tÃ¬m tháº¥y file .pkl nÃ o trong ./results')
else:
    for p in pkl_files:
        print(f'- {p.name} | size={p.stat().st_size:,} bytes')


In [ ]:
from pathlib import Path

expected_models = {
    'random_forest_model.pkl',
    'xgboost_model.pkl',
    'lightgbm_model.pkl',
    'stacking_model.pkl',
}
existing = {p.name for p in Path('./results').glob('*.pkl')}
missing = sorted(expected_models - existing)

print('======= KIEM TRA MODEL DA LUU =======')
print('Existing .pkl files:', sorted(existing))
if missing:
    print('Missing expected models:', missing)
else:
    print('All expected model files are present.')


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

print('======= PREDICT HOUSE PRICE WITH TRAINED MODELS =======')

# Prefer stacking, fallback to random forest
model_path = None
for candidate in ['./results/stacking_model.pkl', './results/random_forest_model.pkl']:
    if os.path.exists(candidate):
        model_path = candidate
        break

if model_path is None:
    raise FileNotFoundError('No trained model found in ./results/. Run the training cells first.')

model = joblib.load(model_path)
print('Loaded model:', model_path)

# Build a one-row input using the same feature columns as X_train
feature_cols = list(X_train.columns)
input_row = pd.DataFrame([{col: 0 for col in feature_cols}])

# Example values - chá»‰nh láº¡i cÃ¡c giÃ¡ trá»‹ nÃ y Ä‘á»ƒ dá»± Ä‘oÃ¡n nhÃ  khÃ¡c
sample_values = {
    'Area': 60,
    'Bedrooms': 2,
    'Bathrooms': 1,
    'Floors': 1,
    'Alley Width': 4,
    'Area_per_Bedroom': 60 / 2,
    'Is_Large_House': 0,
}

# Fill numeric columns if they exist
for col, value in sample_values.items():
    if col in input_row.columns:
        input_row.loc[0, col] = value

# Set district example
district_example = 'District_Quáº­n 1'
if district_example in input_row.columns:
    input_row.loc[0, district_example] = 1

# Set property type example
property_type_example = 'Property Type_NhÃ  riÃªng'
if property_type_example in input_row.columns:
    input_row.loc[0, property_type_example] = 1

# Align to the model if it exposes feature names
if hasattr(model, 'feature_names_in_'):
    expected = list(model.feature_names_in_)
    for col in expected:
        if col not in input_row.columns:
            input_row[col] = 0
    input_row = input_row[[col for col in expected if col in input_row.columns]]

try:
    raw_pred = model.predict(input_row)
except Exception:
    raw_pred = model.predict(input_row.values)

pred_price = np.expm1(raw_pred).ravel()[0]

print('Predicted price (Triá»‡u VNÄ):', round(pred_price, 2))
print('Predicted price (Tá»· VNÄ):', round(pred_price / 1000, 2))

display(input_row.T)